# Análise de codificação e balanceamento

Validação: base v1, 3.337 instâncias, `StratifiedGroupKFold` com 5 folds, grupos por posto (`Código`) e seed 42. Os resultados abaixo vêm dos experimentos executados no notebook de balanceamento; a seleção e avaliação usam os mesmos folds, portanto são exploratórios.

## O que o código principal faz

Em [`src/experimentos.py`](../src/experimentos.py), `X` contém os 12 totais mensais de chuva, que já são numéricos; não é necessário aplicar encoding a essas colunas. O ano fica fora de `X`. O alvo `y` continua com rótulos textuais de mesorregião, que o `KNeighborsClassifier` aceita diretamente — não é necessário usar `LabelEncoder` no alvo para treinar o modelo.

O modelo principal também não reamostra as classes. Usa imputação pela mediana, scaler e KNN dentro do pipeline, com F1 macro como métrica principal. Essa abordagem preserva a distribuição real dos dados de treino e serve como baseline para comparar alternativas.

## Trecho da pipeline principal

A preparação devolve as 12 colunas mensais, o alvo textual e o código do posto. A configuração da base v1 fica dentro de um `Pipeline` para que imputação e escala sejam ajustadas somente nos dados de treino de cada fold:

```python
X, y, grupos = X_y_grupos(df)  # X: meses numéricos; y: nomes das mesorregiões

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(
        n_neighbors=5, weights='distance', metric='manhattan'
    )),
])
```

Não há encoder em `X`: os meses já são valores quantitativos. Também não há reamostrador nessa pipeline; o KNN recebe os rótulos textuais diretamente.

## Trechos das alternativas avaliadas

Para testar o ano, ele é tratado como categoria no `ColumnTransformer`; `handle_unknown='ignore'` evita erro se surgir no teste um ano que não apareceu no treino:

```python
X_ano = X.copy()
X_ano['Ano'] = df.loc[X.index, 'Ano'].astype(str)

prep_ano = ColumnTransformer([
    ('meses', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), MESES),
    ('ano', OneHotEncoder(handle_unknown='ignore'), ['Ano']),
])
pipe_ano = Pipeline([
    ('preprocessamento', prep_ano),
    ('knn', KNeighborsClassifier(
        n_neighbors=5, weights='distance', metric='manhattan'
    )),
])
```

No teste de SMOTE, a amostragem fica depois da imputação e da escala. Usar `imblearn.pipeline.Pipeline` faz com que `fit_resample` aconteça só ao ajustar cada fold de treino, nunca no conjunto de teste:

```python
pipe_smote = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('balanceamento', SMOTE(k_neighbors=5, random_state=42)),
    ('knn', KNeighborsClassifier(
        n_neighbors=5, weights='distance', metric='manhattan'
    )),
])
```

Esses são trechos ilustrativos das alternativas do experimento; não substituem nem modificam o código principal.

## Comparação dos experimentos

| Configuração | Instâncias | F1 macro | Desvio F1 macro | Precisão SF | Recall SF | F1 SF |
|---|---:|---:|---:|---:|---:|---:|
| Sem reamostragem (base v1) | 3.337 | 0,723 | 0,032 | 0,638 | 0,497 | 0,559 |
| RandomOverSampler | 3.337 | 0,682 | 0,028 | 0,456 | 0,718 | 0,558 |
| RandomUnderSampler | 3.337 | 0,658 | 0,025 | 0,438 | 0,700 | 0,539 |
| SMOTE (`k=3`) | 3.337 | 0,685 | 0,030 | 0,480 | 0,703 | 0,571 |
| SMOTE (`k=5`) | 3.337 | 0,694 | 0,034 | 0,481 | 0,712 | 0,574 |
| 12 meses + Ano em One-Hot | 3.337 | **0,769** | 0,032 | 0,715 | 0,579 | **0,640** |

Nos testes de reamostragem, o imputador e o scaler foram ajustados no treino de cada fold e o sampler foi aplicado somente ao treino, usando `imblearn.pipeline.Pipeline`. O teste manteve a distribuição original. Para a variação com ano, o `ColumnTransformer` manteve imputação e escala nos meses e usou `OneHotEncoder(handle_unknown='ignore')` para o ano.

## Interpretação

- **Codificação dos meses:** a escolha de não codificá-los é adequada: são medidas contínuas em milímetros, não categorias. Transformar meses em números de 1 a 12 criaria uma distância linear artificial, inclusive colocando dezembro longe de janeiro.
- **Alvo:** manter os nomes das mesorregiões como rótulos é válido para o classificador; esses valores não entram no cálculo das distâncias entre amostras.
- **Balanceamento:** a base sem reamostragem teve o maior F1 macro entre as opções balanceadas (0,723 contra 0,694 do melhor, SMOTE `k=5`). Assim, não reamostrar é uma boa escolha quando a prioridade é o desempenho geral equilibrado entre classes.
- **Trade-off do São Francisco:** RandomOverSampler elevou o recall de 0,497 para 0,718, mas a precisão caiu de 0,638 para 0,456 e o F1 macro para 0,682. SMOTE `k=5` teve o maior F1 SF entre os métodos de reamostragem (0,574 versus 0,559 na base), com queda no F1 macro. Portanto, balancear pode fazer sentido se recall for prioridade, mas não é uma melhoria geral.
- **Ano em One-Hot:** a maior pontuação observada foi da variação com ano (F1 macro 0,769; F1 SF 0,640). Isso indica que a exclusão do ano não foi a melhor opção nestes folds; porém, como os mesmos folds foram usados para comparar as alternativas, o resultado precisa de validação independente antes de recomendar a mudança no modelo principal.

## Conclusão

Manter os meses numéricos sem encoding e não reamostrar é uma escolha defensável para a pipeline principal: preserva a representação natural das chuvas e obteve o melhor F1 macro entre os métodos balanceados. Isso não significa que essas escolhas sejam superiores em toda métrica: o recall do São Francisco melhora com reamostragem, e Ano em One-Hot foi melhor neste experimento. Antes de alterar a pipeline, é necessário validar o ganho do ano em dados/folds independentes e decidir se a prioridade é F1 macro geral ou recall da classe minoritária.